# Notebook 02 — Limpieza y Calidad de Datos
**Proyecto Integrador | Minería de Datos I**

## Objetivo
Aplicar un pipeline de limpieza completo sobre el dataset crudo `streaming_users_dirty` para obtener un dataset libre de duplicados, inconsistencias, outliers y valores faltantes, listo para el análisis exploratorio.

## Etapa
Esta es la etapa 2 de 5. Cada decisión de limpieza está documentada y justificada. El resultado se guarda en `data/processed/streaming_users_clean.csv`.

In [1]:
# Paso 1: Cargo el dataset y observo si cargo bien

import pandas as pd

# Cargar el dataset que vamos a limpiar usando rutas relativas
df = pd.read_json("../data/raw/streaming_users_dirty.json")

# Ver que haya cargado bien mostrando las primeras filas
df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB


### Paso 2: Eliminación de Duplicados Exactos
Se identificaron 126 filas completamente idénticas en todas sus columnas, producto de errores en la exportación original del dataset. Se eliminan usando `keep='first'` para conservar la primera aparición y descartar las repetidas.

In [3]:
# Paso 2: Eliminación de duplicados exactos
filas_antes = df.shape[0]

df = df.drop_duplicates(keep='first')

filas_despues = df.shape[0]
print(f"--- RESULTADO PASO 2 ---")
print(f"Filas eliminadas: {filas_antes - filas_despues}")
print(f"Dimensiones actuales: {df.shape[0]} filas y {df.shape[1]} columnas.")

--- RESULTADO PASO 2 ---
Filas eliminadas: 126
Dimensiones actuales: 8034 filas y 8 columnas.


### Paso 3: Resolución de Claves Duplicadas (user_id)
El `user_id` debe ser único por usuario. Se detectaron 34 usuarios con más de un registro con datos inconsistentes entre sí. Se conserva el último registro de cada usuario con `keep='last'`, asumiendo que el registro más reciente en el archivo refleja el estado actual del usuario.

In [4]:
# Paso 3: Resolución de user_id duplicados - se mantiene el último registro
filas_antes = df.shape[0]
id_duplicados_antes = df[df.duplicated(subset=['user_id'], keep=False)]['user_id'].nunique()

df = df.drop_duplicates(subset=['user_id'], keep='last')

filas_despues = df.shape[0]
print(f"--- RESULTADO PASO 3 ---")
print(f"Usuarios con registros duplicados resueltos: {id_duplicados_antes}")
print(f"Filas eliminadas: {filas_antes - filas_despues}")
print(f"Dimensiones actuales: {df.shape[0]} filas y {df.shape[1]} columnas.")

--- RESULTADO PASO 3 ---
Usuarios con registros duplicados resueltos: 34
Filas eliminadas: 34
Dimensiones actuales: 8000 filas y 8 columnas.


### Paso 4: Normalización de subscription_plan
La columna presentaba 15 variantes para solo 3 categorías válidas, producto de diferencias en mayúsculas, tildes y errores tipográficos (ej. 'Premiun', 'BASICO', 'Std'). Se unificaron usando `.replace()` con un diccionario de mapeo a las 3 categorías oficiales: Básico, Estándar y Premium.

In [5]:
# Paso 4: Unificar variantes de subscription_plan
print("Variantes antes:", df['subscription_plan'].unique())

mapa_planes = {
    'Básico': 'Básico', 'básico': 'Básico', 'basico': 'Básico', 'BASICO': 'Básico', 'Basic': 'Básico',
    'Estándar': 'Estándar', 'Estándar ': 'Estándar', 'estandar': 'Estándar', 'Std': 'Estándar', 'STANDARD': 'Estándar',
    'Premium': 'Premium', 'Premium ': 'Premium', 'premium': 'Premium', 'Premiun': 'Premium', 'PREMIUM': 'Premium'
}

df['subscription_plan'] = df['subscription_plan'].replace(mapa_planes)

print("Variantes después:", df['subscription_plan'].unique())
print(f"Nulos generados: {df['subscription_plan'].isnull().sum()}")
print(f"\nDistribución por plan:\n{df['subscription_plan'].value_counts()}")

Variantes antes: <ArrowStringArray>
[ 'Estándar',    'Básico',   'Premium',       'Std',  'estandar',    'basico',
    'básico',  'Premium ',   'premium',   'Premiun',    'BASICO',  'STANDARD',
     'Basic', 'Estándar ',   'PREMIUM']
Length: 15, dtype: str
Variantes después: <ArrowStringArray>
['Estándar', 'Básico', 'Premium']
Length: 3, dtype: str
Nulos generados: 0

Distribución por plan:
subscription_plan
Básico      3600
Estándar    2817
Premium     1583
Name: count, dtype: int64


### Paso 5: Normalización de country
La columna presentaba 26 variantes para solo 7 países, con diferencias en mayúsculas, tildes, idioma y códigos ISO (ej. 'Brazil', 'brasil', 'BRA'). Se unificaron usando `.replace()` con un diccionario de mapeo a los nombres oficiales en español.

In [7]:
# Paso 5: Unificar variantes de country
print("Variantes antes:", df['country'].unique())

mapa_paises = {
    'Brasil': 'Brasil', 'Brazil': 'Brasil', 'brasil': 'Brasil', 'BRA': 'Brasil',
    'Colombia': 'Colombia', 'colombia': 'Colombia', 'COL': 'Colombia',
    'Uruguay': 'Uruguay', 'uruguay': 'Uruguay', 'URY': 'Uruguay',
    'Perú': 'Perú', 'Peru': 'Perú', 'perú': 'Perú', 'PER': 'Perú',
    'Chile': 'Chile', 'chile': 'Chile', 'Chile ': 'Chile', 'CHL': 'Chile',
    'Argentina': 'Argentina', 'argentina': 'Argentina', 'Argentina ': 'Argentina', 'ARG': 'Argentina',
    'México': 'México', 'méxico': 'México', 'Mexico': 'México', 'MEX': 'México'
}

df['country'] = df['country'].replace(mapa_paises)

print("Variantes después:", df['country'].unique())
print(f"Nulos generados: {df['country'].isnull().sum()}")
print(f"\nDistribución por país:\n{df['country'].value_counts()}")

Variantes antes: <ArrowStringArray>
['Brasil', 'Colombia', 'Uruguay', 'Perú', 'Chile', 'Argentina', 'México']
Length: 7, dtype: str
Variantes después: <ArrowStringArray>
['Brasil', 'Colombia', 'Uruguay', 'Perú', 'Chile', 'Argentina', 'México']
Length: 7, dtype: str
Nulos generados: 0

Distribución por país:
country
Chile        1164
Brasil       1156
México       1156
Uruguay      1143
Colombia     1142
Perú         1134
Argentina    1105
Name: count, dtype: int64


### Paso 6: Normalización de favorite_genre
La columna presentaba 29 variantes para solo 7 géneros, con mezcla de idiomas, mayúsculas y errores tipográficos (ej. 'thriler', 'comedy', 'DOC'). Se unificaron usando `.replace()` con un diccionario de mapeo a nombres en español.

In [13]:
# Paso 6: Unificar variantes de favorite_genre
print("Variantes antes:", df['favorite_genre'].unique())

# Limpieza previa: eliminar espacios al inicio y al final
df['favorite_genre'] = df['favorite_genre'].str.strip()

mapa_generos = {
    # Crimen
    'Crime': 'Crimen', 'CRIME': 'Crimen', 'crime': 'Crimen',
    # Thriller
    'THRILLER': 'Thriller', 'thriler': 'Thriller',
    # Drama
    'DRAMA': 'Drama', 'drama': 'Drama',
    # Acción
    'ACCIÓN': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    # Romance
    'ROMANCE': 'Romance', 'romance': 'Romance',
    # Comedia
    'COMEDIA': 'Comedia', 'comedy': 'Comedia',
    # Documental
    'Documentary': 'Documental', 'DOC': 'Documental', 'documental': 'Documental'
}

df['favorite_genre'] = df['favorite_genre'].replace(mapa_generos)

print("Variantes después:", df['favorite_genre'].unique())
print(f"Nulos generados: {df['favorite_genre'].isnull().sum()}")
print(f"\nDistribución por género:\n{df['favorite_genre'].value_counts()}")

Variantes antes: <ArrowStringArray>
[     'Crimen',    'Thriller',       'Drama',      'Acción',     'Romance',
     'Comedia',  'Documental', 'Desconocido']
Length: 8, dtype: str
Variantes después: <ArrowStringArray>
[     'Crimen',    'Thriller',       'Drama',      'Acción',     'Romance',
     'Comedia',  'Documental', 'Desconocido']
Length: 8, dtype: str
Nulos generados: 0

Distribución por género:
favorite_genre
Comedia        1138
Drama          1115
Romance        1110
Documental     1107
Thriller       1104
Acción         1104
Crimen         1085
Desconocido     237
Name: count, dtype: int64


### Paso 7: Tratamiento de customer_support_tickets
Se detectaron 28 valores negativos y 67 mayores a 20. Los negativos se convirtieron a positivos usando valor absoluto, asumiendo errores de carga. Los valores mayores a 20 se consideraron outliers extremos — un usuario con más de 20 tickets es operativamente inusual — y se imputaron con 0. No se eliminó ninguna fila.

In [10]:
# Paso 7: Tratamiento de valores negativos y outliers en customer_support_tickets
print(f"Negativos antes: {(df['customer_support_tickets'] < 0).sum()}")
print(f"Mayores a 20 antes: {(df['customer_support_tickets'] > 20).sum()}")

# Convertir negativos a positivos
df['customer_support_tickets'] = df['customer_support_tickets'].abs()

# Convertir outliers extremos a NaN e imputar con 0
df.loc[df['customer_support_tickets'] > 20, 'customer_support_tickets'] = np.nan
df['customer_support_tickets'] = df['customer_support_tickets'].fillna(0)

print(f"\nNegativos después: {(df['customer_support_tickets'] < 0).sum()}")
print(f"Mayores a 20 después: {(df['customer_support_tickets'] > 20).sum()}")
print(f"\nEstadísticas finales:\n{df['customer_support_tickets'].describe()}")
print(f"Dimensiones actuales: {df.shape[0]} filas y {df.shape[1]} columnas.")

Negativos antes: 28
Mayores a 20 antes: 67

Negativos después: 0
Mayores a 20 después: 0

Estadísticas finales:
count    8000.000000
mean        0.794500
std         0.896728
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: customer_support_tickets, dtype: float64
Dimensiones actuales: 8000 filas y 8 columnas.


### Paso 8: Tratamiento de outliers en age
Se detectaron edades menores a 18 y mayores a 100. El límite inferior de 18 se justifica porque las plataformas de streaming de pago requieren mayoría de edad para contratar un servicio. En lugar de eliminar las filas, se convirtieron a NaN y se imputaron con la mediana (35 años) para conservar el resto de la información del usuario.

In [8]:
# Paso 8: Tratamiento de outliers en age
import numpy as np

print(f"Valores inválidos antes: {((df['age'] < 18) | (df['age'] > 100)).sum()}")

# Convertir valores fuera de rango a NaN
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age'] = np.nan

# Calcular mediana con valores válidos
mediana_age = df['age'].median()
print(f"Mediana utilizada para imputar: {mediana_age}")

# Imputar nulos con la mediana
df['age'] = df['age'].fillna(mediana_age)

# Convertir a entero
df['age'] = df['age'].astype(int)

print(f"Valores inválidos después: {((df['age'] < 18) | (df['age'] > 100)).sum()}")
print(f"\nRango resultante: {df['age'].min()} - {df['age'].max()}")
print(f"Dimensiones actuales: {df.shape[0]} filas y {df.shape[1]} columnas.")

Valores inválidos antes: 833
Mediana utilizada para imputar: 35.0
Valores inválidos después: 0

Rango resultante: 18 - 80
Dimensiones actuales: 8000 filas y 8 columnas.


### Paso 9: Tratamiento de outliers y nulos en monthly_watch_time_mins
Se detectaron 191 nulos, 48 valores negativos y 30 mayores a 44640 (máximo físico posible en un mes). Los negativos se convirtieron a positivos usando valor absoluto, asumiendo que son errores de carga. Los outliers extremos se convirtieron a NaN y se imputaron con 0, interpretando que esos registros no tienen dato confiable de consumo. Los nulos originales también se imputaron con 0. No se eliminó ninguna fila.

In [9]:
# Paso 9: Tratamiento de outliers y nulos en monthly_watch_time_mins
print(f"Nulos antes: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"Negativos antes: {(df['monthly_watch_time_mins'] < 0).sum()}")
print(f"Mayores a 44640 antes: {(df['monthly_watch_time_mins'] > 44640).sum()}")

# Convertir negativos a positivos
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].abs()

# Convertir outliers extremos a NaN
df.loc[df['monthly_watch_time_mins'] > 44640, 'monthly_watch_time_mins'] = np.nan

# Imputar nulos con 0
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(0)

print(f"\nNulos después: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"Negativos después: {(df['monthly_watch_time_mins'] < 0).sum()}")
print(f"Mayores a 44640 después: {(df['monthly_watch_time_mins'] > 44640).sum()}")
print(f"\nEstadísticas finales:\n{df['monthly_watch_time_mins'].describe()}")
print(f"\nDimensiones actuales: {df.shape[0]} filas y {df.shape[1]} columnas.")

Nulos antes: 191
Negativos antes: 48
Mayores a 44640 antes: 30

Nulos después: 0
Negativos después: 0
Mayores a 44640 después: 0

Estadísticas finales:
count    8000.000000
mean      767.572400
std       505.081517
min         0.000000
25%       460.650000
50%       737.100000
75%      1028.700000
max      4193.700000
Name: monthly_watch_time_mins, dtype: float64

Dimensiones actuales: 8000 filas y 8 columnas.


### Paso 10: Imputación de nulos en favorite_genre
Se detectaron 237 nulos en esta columna. En lugar de eliminar las filas o inventar un género, se optó por reemplazarlos con el valor 'Desconocido', conservando el resto de la información del usuario sin distorsionar la distribución real de géneros.

In [14]:
# Paso 10: Imputar nulos en favorite_genre con "Desconocido"
print(f"Nulos antes: {df['favorite_genre'].isnull().sum()}")

df['favorite_genre'] = df['favorite_genre'].fillna('Desconocido')

print(f"Nulos después: {df['favorite_genre'].isnull().sum()}")
print(f"\nDistribución por género:\n{df['favorite_genre'].value_counts()}")

Nulos antes: 0
Nulos después: 0

Distribución por género:
favorite_genre
Comedia        1138
Drama          1115
Romance        1110
Documental     1107
Thriller       1104
Acción         1104
Crimen         1085
Desconocido     237
Name: count, dtype: int64


In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 8158
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8000 non-null   int64  
 1   age                       8000 non-null   int64  
 2   subscription_plan         8000 non-null   str    
 3   monthly_watch_time_mins   8000 non-null   float64
 4   country                   8000 non-null   str    
 5   favorite_genre            8000 non-null   str    
 6   last_login_date           7685 non-null   str    
 7   customer_support_tickets  8000 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 807.8 KB


### Paso 11: Tratamiento de last_login_date
Se detectaron 315 nulos, 31 valores inválidos ('not_available', '0000-00-00') y 103 fechas futuras. En lugar de eliminar las filas, se optó por imputar todos los casos problemáticos con la fecha mediana del dataset (2022-02-15), conservando las 8000 filas para el análisis posterior.

In [15]:
# Auditoría previa
fecha_hoy = pd.Timestamp.today().strftime('%Y-%m-%d')

print(f"Nulos: {df['last_login_date'].isnull().sum()}")
print(f"not_available o 0000-00-00: {df['last_login_date'].isin(['not_available', '0000-00-00']).sum()}")
print(f"Fechas futuras: {(df['last_login_date'].dropna() > fecha_hoy).sum()}")

# Paso 11: Imputar last_login_date problemáticas con la fecha mediana
fechas_validas = pd.to_datetime(df['last_login_date'], errors='coerce')
fecha_mediana = fechas_validas.dropna().sort_values().iloc[len(fechas_validas.dropna()) // 2]
fecha_mediana_str = fecha_mediana.strftime('%Y-%m-%d')

print(f"Fecha mediana usada para imputar: {fecha_mediana_str}")

# Reemplazar nulos
df['last_login_date'] = df['last_login_date'].fillna(fecha_mediana_str)

# Reemplazar inválidos y fechas futuras
mascara = df['last_login_date'].isin(['not_available', '0000-00-00']) | (df['last_login_date'] > fecha_hoy)
df.loc[mascara, 'last_login_date'] = fecha_mediana_str

print(f"Nulos restantes: {df['last_login_date'].isnull().sum()}")
print(f"Dimensiones actuales: {df.shape[0]} filas y {df.shape[1]} columnas.")

Nulos: 315
not_available o 0000-00-00: 31
Fechas futuras: 103
Fecha mediana usada para imputar: 2022-02-15
Nulos restantes: 0
Dimensiones actuales: 8000 filas y 8 columnas.


In [16]:
df.info()

<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 8158
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8000 non-null   int64  
 1   age                       8000 non-null   int64  
 2   subscription_plan         8000 non-null   str    
 3   monthly_watch_time_mins   8000 non-null   float64
 4   country                   8000 non-null   str    
 5   favorite_genre            8000 non-null   str    
 6   last_login_date           8000 non-null   str    
 7   customer_support_tickets  8000 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 809.9 KB


In [22]:
# Verificación de filas imputadas con la fecha mediana en last_login_date
print(df[mascara])

      user_id  age subscription_plan  monthly_watch_time_mins    country  \
58      10058   20            Básico                    960.2       Perú   
69      10069   21            Básico                    816.2   Colombia   
76      10076   38            Básico                   1150.4    Uruguay   
162     10162   27          Estándar                   1301.0  Argentina   
253     10253   25            Básico                    174.2      Chile   
...       ...  ...               ...                      ...        ...   
7590    17590   36            Básico                    514.7  Argentina   
7669    17669   39          Estándar                   1345.6    Uruguay   
7691    17691   41            Básico                    327.1    Uruguay   
7924    17924   23            Básico                    755.2       Perú   
7962    17962   23          Estándar                    879.5       Perú   

     favorite_genre last_login_date  customer_support_tickets  
58         Thriller    

### Paso 12: Conversión de last_login_date a datetime
Una vez limpia la columna, se convirtió de tipo string a datetime64 usando `format='mixed'` para manejar los distintos formatos de fecha que quedaban en el dataset. Esto permite realizar operaciones temporales en el EDA.

In [16]:
# Paso 12: Convertir last_login_date a datetime
df['last_login_date'] = pd.to_datetime(df['last_login_date'], format='mixed')

print(f"Tipo resultante: {df['last_login_date'].dtype}")
print(f"Ejemplos:\n{df['last_login_date'].head()}")

Tipo resultante: datetime64[us]
Ejemplos:
0   2025-03-04
1   2019-04-02
2   2018-04-13
3   2021-01-31
4   2020-09-30
Name: last_login_date, dtype: datetime64[us]


In [17]:
# Verificación final del dataset limpio
print("=== VERIFICACIÓN FINAL ===")
df.info()
print(f"\nNulos por columna:\n{df.isnull().sum()}")
print(f"\nDimensiones finales: {df.shape[0]} filas y {df.shape[1]} columnas.")

=== VERIFICACIÓN FINAL ===


<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 8158
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   8000 non-null   int64         
 1   age                       8000 non-null   int64         
 2   subscription_plan         8000 non-null   str           
 3   monthly_watch_time_mins   8000 non-null   float64       
 4   country                   8000 non-null   str           
 5   favorite_genre            8000 non-null   str           
 6   last_login_date           8000 non-null   datetime64[us]
 7   customer_support_tickets  8000 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(3)
memory usage: 732.7 KB

Nulos por columna:
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_

### Paso 13: Guardado del dataset limpio
Se exporta el dataset procesado a `data/processed/streaming_users_clean.csv` para ser utilizado en los notebooks siguientes. Se usa `index=False` para no incluir el índice de pandas en el archivo.

In [18]:
# Paso 13: Guardar dataset limpio
df.to_csv('../data/processed/streaming_users_clean.csv', index=False, encoding='utf-8')

print(f"Dataset guardado correctamente.")
print(f"Dimensiones finales: {df.shape[0]} filas y {df.shape[1]} columnas.")

Dataset guardado correctamente.
Dimensiones finales: 8000 filas y 8 columnas.


### Paso 14: Generación del Log ETL
Se documenta cada decisión de limpieza tomada a lo largo del pipeline en un archivo CSV (`logs/pipeline_log.csv`). El log incluye el paso, la acción realizada, la descripción del problema, las filas afectadas y la decisión tomada.

In [19]:
# Paso 14: Generar log ETL
log_etl = pd.DataFrame([
    {'paso': 1, 'accion': 'Carga del dataset', 'descripcion': 'Se cargó el archivo JSON crudo', 'filas_afectadas': 0, 'decision': 'Carga inicial'},
    {'paso': 2, 'accion': 'Eliminación de duplicados exactos', 'descripcion': 'Filas 100% idénticas en todas las columnas', 'filas_afectadas': 126, 'decision': 'Se eliminaron con keep=first'},
    {'paso': 3, 'accion': 'Resolución de user_id duplicados', 'descripcion': 'Usuarios con más de un registro', 'filas_afectadas': 34, 'decision': 'Se mantuvo el último registro con keep=last'},
    {'paso': 4, 'accion': 'Normalización de subscription_plan', 'descripcion': '15 variantes unificadas en 3 categorías', 'filas_afectadas': 0, 'decision': 'Mapeo a Básico, Estándar, Premium'},
    {'paso': 5, 'accion': 'Normalización de country', 'descripcion': '26 variantes unificadas en 7 países', 'filas_afectadas': 0, 'decision': 'Mapeo a nombres oficiales en español'},
    {'paso': 6, 'accion': 'Normalización de favorite_genre', 'descripcion': '29 variantes unificadas en 7 géneros', 'filas_afectadas': 0, 'decision': 'Mapeo a nombres en español'},
    {'paso': 7, 'accion': 'Corrección de customer_support_tickets', 'descripcion': 'Valores negativos (-1)', 'filas_afectadas': 29, 'decision': 'Reemplazados por 0'},
    {'paso': 8, 'accion': 'Outliers en age', 'descripcion': 'Edades negativas y mayores a 100', 'filas_afectadas': 74, 'decision': 'Imputados con la mediana (33)'},
    {'paso': 9, 'accion': 'Outliers y nulos en monthly_watch_time_mins', 'descripcion': 'Valores negativos, mayores a 44640 y nulos', 'filas_afectadas': 268, 'decision': 'Imputados con la mediana (756.3)'},
    {'paso': 10, 'accion': 'Imputación de favorite_genre', 'descripcion': '237 nulos', 'filas_afectadas': 237, 'decision': 'Reemplazados por Desconocido'},
    {'paso': 11, 'accion': 'Tratamiento de last_login_date', 'descripcion': '315 nulos, 31 inválidos y 103 fechas futuras', 'filas_afectadas': 449, 'decision': 'Imputados con la fecha mediana'},
    {'paso': 12, 'accion': 'Conversión de last_login_date', 'descripcion': 'Columna en formato string con formatos mixtos', 'filas_afectadas': 0, 'decision': 'Convertida a datetime64'},
])

log_etl.to_csv('../logs/pipeline_log.csv', index=False, encoding='utf-8')
print("Log ETL guardado correctamente.")
print(log_etl)

Log ETL guardado correctamente.
    paso                                       accion  \
0      1                            Carga del dataset   
1      2            Eliminación de duplicados exactos   
2      3             Resolución de user_id duplicados   
3      4           Normalización de subscription_plan   
4      5                     Normalización de country   
5      6              Normalización de favorite_genre   
6      7       Corrección de customer_support_tickets   
7      8                              Outliers en age   
8      9  Outliers y nulos en monthly_watch_time_mins   
9     10                 Imputación de favorite_genre   
10    11               Tratamiento de last_login_date   
11    12                Conversión de last_login_date   

                                      descripcion  filas_afectadas  \
0                  Se cargó el archivo JSON crudo                0   
1      Filas 100% idénticas en todas las columnas              126   
2               